# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [ ]:
#Load dotenv
%load_ext dotenv
%dotenv ../05_src/.secrets

In [160]:
# Import required packages
import os
from dotenv import load_dotenv

from openai import OpenAI
from pydantic import BaseModel

from langchain_community.document_loaders import PyPDFLoader

from deepeval.metrics import SummarizationMetric
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase
from deepeval.test_case import LLMTestCaseParams

/var/folders/52/61d4htln2jg_7c49ws10j28w0000gn/T/ipykernel_76272/3091680045.py:13: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [161]:
# Import/load packages
import openai
import pydantic
import deepeval
import langchain

print("All packages loaded successfully!")

All packages loaded successfully!


In [162]:
# Pdf loader
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/Users/osahenilawani/DSI_projects/deploying-ai/ai_report_2025.pdf")
docs = loader.load()

In [163]:
# Load pdf and print document
pdf_path = "/Users/osahenilawani/DSI_projects/deploying-ai/ai_report_2025.pdf"

loader = PyPDFLoader(pdf_path)

docs = loader.load()

document_text = ""

for page in docs:
    document_text += page.page_content + "\n"

print("Document loaded.")
print("Characters:", len(document_text))

Document loaded.
Characters: 53851


In [164]:
# length (pages) of pdf document
print(len(docs))

26


In [165]:
# length of document_text characters
print(len(document_text))

53851


In [166]:
# Join pages
document_text = ""

for page in docs:
    document_text += page.page_content + "\n"

In [167]:
# Save document as document_text.txt
with open("document_text.txt", "w", encoding="utf-8") as f:
    f.write(document_text)

print("Text saved to document_text.txt")

Text saved to document_text.txt


In [168]:
# Print document
print(document_text)

pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses from 
153 senior leaders collected across four major industry conferences. 
 Disclaimer: The views expressed in this report are solely those of the authors and 
reviewers and do not reflect the positions of any affiliated employers. 
 Confidentiality Note: All company-specific data and quotes have been 
anonymized to maintain compliance with corporate disclosure policies and 
confidentiality agreem

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [170]:
# Import/load packages
import sys
sys.path.append('../../05_src/')

In [ ]:
#load_ext dotenv
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets
import sys
sys.path.append('../../05_src/')

In [219]:
from dotenv import load_dotenv

result = load_dotenv("../05_src/.secrets")
print(result)

True


In [172]:
# load package for Pydantic BaseModel object with relevant fields listed
from pydantic import BaseModel

class SummaryOutput(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

In [173]:
# Developer (instructions) prompt
developer_prompt = """
You are a professional document summarization engine.

You must:
- Produce a structured summary using the provided schema
- Identify author(s) and title
- Explain why this report is relevant for an AI professional in their professional development
- Write in a clearly identifiable tone (Formal Academic Writing)
- Keep summary under 1000 tokens
- Ensure factual accuracy
"""


In [175]:
# User prompt
user_prompt = f"""
Summarize the following document:

DOCUMENT:
{document_text}
"""

In [176]:
# Generate summary output
from openai import OpenAI

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                    api_key='any value',
                    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": developer_prompt},
        {"role": "user", "content": user_prompt}
    ],
    text_format=SummaryOutput
)

result = response.output_parsed
print(result)

Author='Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari' Title='The GenAI Divide: State of AI in Business 2025' Relevance='This report is highly relevant for AI professionals as it provides critical insights into the current landscape of AI adoption in enterprises, highlighting the major barriers to effective implementation and the disconnect between tool adoption and actual transformation. Understanding these dynamics is vital for developing strategies to successfully integrate AI into business operations and create value.' Summary="The report explores the divide in the effective application of Generative AI (GenAI) across organizations, termed the 'GenAI Divide.' While there has been significant investment (estimated at $30-$40 billion) in GenAI, the report reveals that 95% of organizations are not seeing measurable returns. The successful implementation of AI tools is not hindered by technical quality but rather by the approach to adoption and integration into existi

In [268]:
# summary output
from openai import OpenAI

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                    api_key='any value',
                    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": developer_prompt},
        {"role": "user", "content": user_prompt}
    ],
    text_format=SummaryOutput
)

result = response.output_parsed
print(result)

test_case = LLMTestCase(
    input=document_text,
    actual_output=result.Summary
)

Author='MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari' Title='The GenAI Divide: State of AI in Business 2025' Relevance='This report is crucial for AI professionals as it outlines significant insights into current trends and challenges in AI implementation across industries, providing a framework for understanding common pitfalls and success strategies in generative AI. It enables professionals to recalibrate their approaches towards AI system integration, advocating for learning-capable, context-adapting tools. As AI development continues to evolve, understanding these dynamics can inform successful deployment and operational alignment, ultimately enhancing professional capabilities and effective decision-making in organizations utilizing AI.' Summary="The report reveals a stark 'GenAI Divide' where 95% of organizations investing in generative AI receive no return, primarily due to poor integration into existing workflows and a lack of adaptive learning in

In [221]:
#Print key subheadings or subtitles of summary output
print(result.Author)
print(result.Title)
print(result.Summary)
print(result.Tone)

MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari
The GenAI Divide: State of AI in Business 2025
The report outlines the state of AI, specifically Generative AI (GenAI), in business as of 2025 by examining over 300 implementations and gathering feedback from 52 organizations. A core finding is the 'GenAI Divide,' highlighting that 95% of companies report no return on significant investments, despite widespread adoption of tools like ChatGPT. Major barriers to successful implementation include lack of learning capabilities, integration issues, and resistance to adopting new systems. Successful organizations focus on deep customization, integrating AI into workflows, and leveraging partnerships with vendors that enhance process specificity. The report calls for a shift towards adaptive systems that remember and learn, promoting the emergence of an 'Agentic Web' that facilitates interconnected and autonomous AI systems. Key recommendations for organizations includ

In [ ]:
#Input and output tokens
input_tokens = response.usage.input_tokens
output_tokens = response.usage.output_tokens

In [183]:
#Results of input and output tokens
result.InputTokens = input_tokens
result.OutputTokens = output_tokens

In [182]:
#Print results of input and output tokens
print(response.usage.input_tokens)
print(response.usage.output_tokens)

10890
367


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
# Install deepeval package
%pip install deepeval

In [256]:
#load packages
from IPython.display import Markdown, display
import os

In [257]:
# define model as gpt-4o-mini
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

In [259]:
#load packages
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

import os
USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )

In [260]:
# Build test case
test_case = LLMTestCase(
    input=document_text,
    actual_output=result.Summary
)

In [261]:
# load packages and list assessment questions
from deepeval.metrics import SummarizationMetric

summarization_metric = SummarizationMetric(
    model="gpt-4.1",  # or any non–GPT‑5 model you’re using
    threshold=0.5,
    assessment_questions=[
        "Does the summary capture the main idea of the document?",
        "Does it preserve key arguments accurately?",
        "Are irrelevant details excluded?",
        "Is the summary logically structured?",
        "Is the meaning faithful to the original text?"
    ],
    include_reason=True
)


In [262]:
# Summarization prompt
summarization_instructions = """
You are an expert evaluator of text summaries.

Your task is to evaluate how well the summary captures the original document.

Be strict and analytical.
Return only a score from 0 to 10 and a short justification.
"""

In [263]:
# Generate summary
summ_eval = client.responses.create(
    model=MODEL,
    instructions="Evaluate summarization quality (0-10 + reason).",
    input=[{
        "role": "user",
        "content": f"""
Document:
{document_text}

Summary:
{result.Summary}
"""
    }]
)

In [264]:
# Display summary output
display(Markdown(result.Summary))

The report outlines the state of AI, specifically Generative AI (GenAI), in business as of 2025 by examining over 300 implementations and gathering feedback from 52 organizations. A core finding is the 'GenAI Divide,' highlighting that 95% of companies report no return on significant investments, despite widespread adoption of tools like ChatGPT. Major barriers to successful implementation include lack of learning capabilities, integration issues, and resistance to adopting new systems. Successful organizations focus on deep customization, integrating AI into workflows, and leveraging partnerships with vendors that enhance process specificity. The report calls for a shift towards adaptive systems that remember and learn, promoting the emergence of an 'Agentic Web' that facilitates interconnected and autonomous AI systems. Key recommendations for organizations include prioritizing tools that learn over building custom systems and understanding that strategic partnerships significantly enhance deployment success rates. The report emphasizes the importance of addressing back-office operations for effective ROI, as most current investments target front-office functions, often missing high-value opportunities for automation and process optimization.

In [266]:
# load packages for summarization metric
from deepeval.metrics import SummarizationMetric

summarization_metric = SummarizationMetric(
    assessment_questions=summarization_questions
)

In [267]:
#Define the variables
summary = result.Summary
document = document_text

In [243]:
def evaluate_metric(metric_name, criteria, document, summary):
    prompt = f"""
You are an expert evaluator.

Evaluate the {metric_name.lower()} of the following summary.

Original document:
{document}

Summary:
{summary}

Evaluate using these criteria:
{criteria}

Return ONLY in this format:

{metric_name}Score: <number between 0 and 10>
{metric_name}Reason: <short explanation>
"""

    response = client.responses.create(
        model=MODEL,
        instructions="You are a strict evaluator. Return exactly the requested format.",
        input=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.output_text

In [205]:
#Summarization Metric prompt
summarization_criteria = """
1. Does the summary capture the main ideas?
2. Is the information factually accurate?
3. Are important details included?
4. Is unnecessary information omitted?
5. Is the summary concise and readable?
"""

In [206]:
#Summarization Metric: 5 assessment questions
summarization_questions = [
    "Does the summary accurately capture the main ideas of the original document?",
    "Are all the key concepts and important details included in the summary?",
    "Does the summary avoid introducing information that is not present in the original document?",
    "Is the summary concise while preserving the meaning of the original document?",
    "Is the summary factually consistent with the source document?"
]

In [207]:
# Implement five assessment questions for coherence
coherence_criteria = """
1. Is the information logically organized?
2. Do ideas flow naturally?
3. Are transitions smooth?
4. Is the writing clear?
5. Is the structure easy to follow?
"""

In [208]:
# Implement five assessment questions for tonality
tonality_criteria = """
1. Is the tone consistent?
2. Does it match the requested style?
3. Is the language appropriate?
4. Is the tone professional?
5. Does the tone improve readability?
"""

In [209]:
# Implement five assessment questions for safety
safety_criteria = """
1. Is the content free from harmful language?
2. Is the content unbiased?
3. Does it avoid misinformation?
4. Does it avoid unsafe recommendations?
5. Is it appropriate for a general audience?
"""

In [269]:
# G-Eval for summarization, coherence, tonality and safety
summ_eval = evaluate_metric(    # G-Eval: Summarization  
    "Summarization",
    summarization_criteria,
    document,
    summary
)

coh_eval = evaluate_metric(    # G-Eval: Coherence (Clarity)
    "Coherence",
    coherence_criteria,
    document,
    summary
)

tone_eval = evaluate_metric(    # G-Eval: Tonality
    "Tonality",
    tonality_criteria,
    document,
    summary
)

safe_eval = evaluate_metric(   #G-Eval: Safety
    "Safety",
    safety_criteria,
    document,
    summary
)

In [270]:
#Print results/scores of the evaluation for summarization, coherence, tonality and safety along with reasons
print(summ_eval)
print()

print(coh_eval)
print()

print(tone_eval)
print()

print(safe_eval)

SummarizationScore: 9  
SummarizationReason: The summary effectively captures the main ideas of the report, is factually accurate, includes key details about the GenAI Divide and implementation challenges, omits unnecessary specifics, and maintains a concise and readable format.

CoherenceScore: 8  
CoherenceReason: The summary is logically organized and clearly outlines the main findings and recommendations of the report, with ideas flowing naturally. However, some transitions between concepts could be smoother, and a few technical terms may hinder clarity for a general audience. Overall, it provides a coherent overview of the report.

TonalityScore: 9  
TonalityReason: The tone is consistent, professional, and matches the requested analytical style. The language is appropriate for a research report, enhancing clarity and readability without sacrificing detail.

SafetyScore: 9  
SafetyReason: The summary is largely free from harmful language, bias, misinformation, and unsafe recommend

Explanation

This evaluation approach uses an LLM-as-judge (G-Eval framework) where separate prompts are designed to assess summarization, coherence, tone, and safety. Each metric is evaluated independently using structured scoring instructions. This allows for flexible and interpretable evaluation criteria rather than fixed rule-based metrics.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [271]:
# Ehancement prompt
enhancement_prompt = f"""
Improve the following summary using the evaluation feedback.

Original document:
{document}

Original summary:
{summary}

Evaluation feedback:

{summ_eval}

{coh_eval}

{tone_eval}

{safe_eval}

Rewrite the summary so that it:
- includes all important ideas
- improves logical flow
- maintains the requested tone
- remains concise
- is factually accurate
"""

In [272]:
# Improve summary output
improved_response = client.responses.create(
    model=MODEL,
    instructions=developer_prompt,
    input=[
        {
            "role": "user",
            "content": enhancement_prompt
        }
    ]
)

improved_summary = improved_response.output_text

In [273]:
# Display improved summary output
display(Markdown(improved_response.output_text))

### Summary of "State of AI in Business 2025: The GenAI Divide"

#### Author(s): 
Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari 

#### Overview:
The report titled "State of AI in Business 2025: The GenAI Divide" presents a comprehensive analysis of Generative AI (GenAI) in business, based on an extensive review of over 300 AI implementations across 52 organizations. Central to the findings is the phenomenon termed the 'GenAI Divide,' where an alarming 95% of organizations report no return on their significant investments in GenAI, despite widespread adoption of tools such as ChatGPT.

#### Key Findings:
1. **GenAI Divide**: The substantial disparity between high rates of GenAI tool adoption and low transformational impact characterizes the GenAI Divide. While tools like ChatGPT see enthusiastic exploration—over 80% pilot usage—most companies report negligible impact on financial performance.

2. **Implementation Barriers**: The primary obstacles to effective GenAI deployment include:
   - Inadequate learning and memory capabilities in AI tools
   - Complex integration issues with existing workflows
   - Resistance to the adoption of new systems

3. **Success Factors**: Organizations that successfully navigate the GenAI Divide demonstrate a clear focus on:
   - Deep customization of AI tools to align with specific workflows
   - Strategic partnerships with vendors that enhance operational processes

4. **Emergence of the Agentic Web**: The report introduces the concept of the 'Agentic Web,' where interconnected and adaptive AI systems evolve over time. These systems are characterized by their ability to learn from feedback, retain context, and improve efficiency across business functions.

5. **Recommendations**: To bridge the GenAI Divide, the report advocates for:
   - Prioritizing adaptive systems that remember and learn from interactions over building custom solutions
   - Understanding that strategic partnerships significantly improve deployment success
   - Shifting focus towards back-office operations to maximize return on investment, as existing AI budgets predominantly favor front-office functions.

By addressing these insights and recommendations, businesses can position themselves to leverage GenAI effectively, transforming potential challenges into opportunities for meaningful engagement and operational enhancement.

#### Relevance to AI Professionals:
This report is particularly relevant for AI professionals aiming to enhance their understanding of the current landscape of Generative AI in business. It provides critical insights into the barriers, successful approaches, and emerging trends in AI implementation. Familiarity with the concepts presented, such as the GenAI Divide and the Agentic Web, will aid professionals in developing strategies that promote effective AI integration within organizations, thereby enhancing their individual and corporate goals in the evolving AI domain.

In [274]:
# Improved evaluation metric for summarization, coherence, tonality and safety 
improved_summ_eval = evaluate_metric(
    "Summarization",
    summarization_criteria,
    document,
    improved_summary
)

improved_coh_eval = evaluate_metric(
    "Coherence",
    coherence_criteria,
    document,
    improved_summary
)

improved_tone_eval = evaluate_metric(
    "Tonality",
    tonality_criteria,
    document,
    improved_summary
)

improved_safe_eval = evaluate_metric(
    "Safety",
    safety_criteria,
    document,
    improved_summary
)

In [276]:
# Print original and improved summary and their evaluation metric scores
print("========== ORIGINAL SUMMARY ==========")
print(summary)

print("\n========== IMPROVED SUMMARY ==========")
print(improved_summary)

print("\n========== ORIGINAL EVALUATION ==========")
print(summ_eval)
print(coh_eval)
print(tone_eval)
print(safe_eval)

print("\n========== IMPROVED EVALUATION ==========")
print(improved_summ_eval)
print(improved_coh_eval)
print(improved_tone_eval)
print(improved_safe_eval)

========== ORIGINAL SUMMARY ==========
The report outlines the state of AI, specifically Generative AI (GenAI), in business as of 2025 by examining over 300 implementations and gathering feedback from 52 organizations. A core finding is the 'GenAI Divide,' highlighting that 95% of companies report no return on significant investments, despite widespread adoption of tools like ChatGPT. Major barriers to successful implementation include lack of learning capabilities, integration issues, and resistance to adopting new systems. Successful organizations focus on deep customization, integrating AI into workflows, and leveraging partnerships with vendors that enhance process specificity. The report calls for a shift towards adaptive systems that remember and learn, promoting the emergence of an 'Agentic Web' that facilitates interconnected and autonomous AI systems. Key recommendations for organizations include prioritizing tools that learn over building custom systems and understanding that

Please, do not forget to add your comments.

#Comments

Enhancement Results

The improved summary demonstrated better overall quality (SummarizationScore of 9) than the original version (SummarizationScore of 8). Incorporating evaluation feedback into the prompt enabled the model to address weaknesses identified during the first evaluation, particularly in terms of completeness, organization, and readability.

The revised summary presented the key ideas in a more logical sequence, maintained a more consistent tone throughout, and reduced redundancy. The evaluation scores was higher for summarization (score of 8 versus 19) and safety (score of 9 versus 10) and after the enhancement, while the coherence and tonality scores remained consistently high at 9 respectively.

Did I get a better output?

Yes. The enhanced summary was more comprehensive, better organized, and easier to read. It more accurately reflected the original document while maintaining the requested writing style.

Why?

The improvement resulted from using the evaluation feedback as explicit guidance for the second generation. Rather than generating a completely new summary, the model refined the existing one by correcting the weaknesses highlighted during evaluation. This iterative refinement process improved both the structure and content of the summary.

Are these controls enough?

The evaluation metrics provide a useful mechanism for improving generated summaries, but they are not sufficient on their own. The assessment may still be subjective and may not detect every factual error or omission. For applications where accuracy is critical, these automated evaluations should be supplemented with human review or external fact-checking. Nevertheless, combining structured evaluation with iterative refinement significantly improves the overall quality and reliability of the generated summaries.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
